# Module 3 · Designing Health Tools (45 min)

Tools are plain Python functions registered with the agent via the `@tool` decorator, which
turns the signature + docstring into a JSON schema the model can call. You'll **edit two tools**
in `ha_workshop/student_tools.py` (the path is printed below). `query_steps` is already done as
a second worked example.

In [ ]:
# Make the top-level ha_workshop package importable and anchor relative paths to the repo root.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'healthagent').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('repo root:', ROOT)

In [ ]:
from ha_workshop import workshop_paths, reload_ha_workshop
print('Edit this file for Module 3:')
print('   ', workshop_paths()['student_tools'])

### Worked reference — a retrieval, an analysis, and a visualization tool
Read these three; your blanks follow the same shape.

In [ ]:
import inspect
from healthagent.tools import retrieval, analysis, visualization
print(inspect.getsource(retrieval.query_sleep))

In [ ]:
from healthagent import config
R0, R1 = config.recent_window()
print(retrieval.query_sleep('u01', R0, R1).summary)
print(analysis.describe_metric('u01', 'total_sleep_hours', R0, R1).summary)
print(visualization.plot_timeseries('u01', 'night_screen', R0, R1).summary)

### TODO (in `ha_workshop/student_tools.py`)
**`query_screen_time`** — load total + night screen minutes. The 2-line body:
```python
df = data.load_user_metric(user_id, 'screen_time',
                           ['total_screen_minutes', 'night_screen_minutes'], start_date, end_date)
rows = df.assign(date=df['date'].dt.strftime('%Y-%m-%d')).to_dict('records')
return ToolResult('table', f"{len(df)} days: mean night screen "
                  f"{round(df['night_screen_minutes'].mean(),1)} min.", rows)
```
**`detect_change`** — fill exactly two lines:
```python
recent_mean = float(recent.mean())
baseline_mean = float(base.mean())
```
Edit the file, save, then run the smoke-test cell below (it reloads your file).

In [ ]:
# Smoke test — reloads your edited file and reports status (no error if still a stub).
from healthagent.llm.schemas import TODO_SENTINEL
st, _ = reload_ha_workshop()
R0, R1 = config.recent_window()
for name, call in [('query_screen_time', lambda: st.query_screen_time('u01', R0, R1)),
                   ('detect_change',     lambda: st.detect_change('u01', 'night_screen'))]:
    r = call()
    done = TODO_SENTINEL not in (r.summary or '')
    print(f"{'✓' if done else '⏳'} {name}: {r.summary[:80]}")

### 🚀 Advanced (optional) — add a `query_location` tool

You implemented screen-time and change-detection. The **same pattern exposes any modality.** The
dataset has a `location` table (time at home, places visited, location entropy, distance travelled)
that no tool reads yet.

Below is a complete `query_location`, built exactly like the `query_*` retrieval tools — it uses
`data.load_user_metric(...)` to filter by user **and** inclusive date range (never the whole table).
Read it, run it.

**Your turn:** add `social_app_minutes` to your `query_screen_time`, or expose `query_heart_rate` for
the recent week. (Reference modality tools live in `healthagent/tools/retrieval.py`.)

In [ ]:
# Advanced (optional): a complete extra retrieval tool, same shape as query_sleep / query_screen_time.
from healthagent.tools.registry import tool
from healthagent.llm.schemas import ToolResult
from healthagent import data, config


@tool
def query_location(user_id: str, start_date: str, end_date: str) -> ToolResult:
    """Retrieve daily location features (time at home, places visited, entropy, distance) over an inclusive date range.

    Args:
        user_id: which user, e.g. 'u01'.
        start_date: ISO YYYY-MM-DD (inclusive).
        end_date: ISO YYYY-MM-DD (inclusive); 'this week' = the dataset's final 7 days.
    """
    cols = ['time_at_home_hours', 'num_places_visited', 'location_entropy', 'total_distance_km']
    df = data.load_user_metric(user_id, 'location', cols, start_date, end_date)   # filters user + dates
    rows = df.assign(date=df['date'].dt.strftime('%Y-%m-%d')).to_dict('records')
    mean_home = round(df['time_at_home_hours'].mean(), 1) if len(df) else float('nan')
    return ToolResult(kind='table',
                      summary=f"{len(df)} days {start_date}..{end_date}: mean time at home {mean_home} h.",
                      value=rows)


# Run it for u01's recent week (the @tool decorator still lets you call it directly):
R0, R1 = config.recent_window()
res = query_location('u01', R0, R1)
print(res.summary)
for row in res.value[-3:]:
    print('  ', row)

When both show ✓ you have a registered toolkit. Stuck? Reference solutions are in
`ha_workshop/solutions/` (your instructor can point you there).